# Retail Demand Forecasting — Data Understanding & Cleaning

## Objective

Prepare the raw Online Retail II transaction data for exploratory data analysis and demand forecasting.

The notebook will:

- inspect the structure and quality of the source data
- identify issues that could distort demand
- investigate cancellations, reversals and operational adjustments
- remove duplicate and overlapping records
- retain valid merchandise demand
- export a clean transaction-level dataset for later analysis

The forecasting target in this project is **positive merchandise-order demand over time**.

In [81]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_RAW = Path("../data/raw")
DATA_CLEAN = Path("../data/cleaned")

FILE_PATH = DATA_RAW / "online_retail_II.xlsx"

Using project-relative paths keeps the notebook portable when the repository is cloned or moved.

In [82]:
excel_file = pd.ExcelFile(FILE_PATH)

excel_file.sheet_names

['Year 2009-2010', 'Year 2010-2011']

The workbook contains two annual transaction worksheets. These will initially be loaded separately so their schemas and date ranges can be validated before combining them.

In [85]:
sheets = pd.read_excel(
    FILE_PATH,
    sheet_name=[0, 1]
)

retail_2009_2010 = sheets[0]
retail_2010_2011 = sheets[1]

print("2009-2010:", retail_2009_2010.shape)
print("2010-2011:", retail_2010_2011.shape)

2009-2010: (525461, 8)
2010-2011: (541910, 8)


In [86]:
display(retail_2009_2010.head())

print(retail_2009_2010.dtypes)
print()
print(retail_2010_2011.dtypes)

print()
print("Columns match:",
      retail_2009_2010.columns.equals(retail_2010_2011.columns))

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object

Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object

Columns match: True


Both worksheets contain the same eight fields and share the same schema.

One row represents a **product line within an invoice**, rather than a complete customer order.

For forecasting, the most important fields are:

- `StockCode` — product identifier
- `Quantity` — units ordered
- `InvoiceDate` — transaction timestamp

`Customer ID` is useful for investigation but is not essential for measuring aggregate product demand.

In [87]:
def missing_summary(df):
    return pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2)
    })

display(missing_summary(retail_2009_2010))
display(missing_summary(retail_2010_2011))

,missing_count,missing_pct
Invoice,0,0.00
StockCode,0,0.00
Description,2928,0.56
Quantity,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Customer ID,107927,20.54
Country,0,0.00


,missing_count,missing_pct
Invoice,0,0.00
StockCode,0,0.00
Description,1454,0.27
Quantity,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Customer ID,135080,24.93
Country,0,0.00


Missing values are concentrated in `Customer ID` and `Description`.

Missing customer identifiers do not automatically invalidate a transaction for demand forecasting because the product, quantity and date can still provide evidence of demand.

Missing descriptions will also not be removed automatically where a valid `StockCode` remains available.

In [90]:
display(
    retail_2009_2010[["Quantity", "Price"]].describe()
)

display(
    retail_2010_2011[["Quantity", "Price"]].describe()
)

,Quantity,Price
count,525461.000000,525461.000000
mean,10.337667,4.688834
std,107.424110,146.126914
min,-9600.000000,-53594.360000
25%,1.000000,1.250000
50%,3.000000,2.100000
75%,10.000000,4.210000
max,19152.000000,25111.090000


,Quantity,Price
count,541910.000000,541910.000000
mean,9.552234,4.611138
std,218.080957,96.759765
min,-80995.000000,-11062.060000
25%,1.000000,1.250000
50%,3.000000,2.080000
75%,10.000000,4.130000
max,80995.000000,38970.000000


In [91]:
def transaction_quality_counts(df):
    return pd.Series({
        "negative_quantity": (df["Quantity"] < 0).sum(),
        "zero_quantity": (df["Quantity"] == 0).sum(),
        "negative_price": (df["Price"] < 0).sum(),
        "zero_price": (df["Price"] == 0).sum()
    })

display(transaction_quality_counts(retail_2009_2010))
display(transaction_quality_counts(retail_2010_2011))

negative_quantity    12326
zero_quantity            0
negative_price           3
zero_price            3687
dtype: int64

negative_quantity    10624
zero_quantity            0
negative_price           2
zero_price            2515
dtype: int64

In [92]:
retail_2009_2010.loc[
    retail_2009_2010["Quantity"] < 0,
    ["Invoice", "StockCode", "Description", "Quantity", "Price"]
].head(20)

,Invoice,StockCode,Description,Quantity,Price
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2.95
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,1.65
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,4.25
181,C489449,21896,POTTING SHED TWINE,-6,2.10
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2.95
183,C489449,21871,SAVE THE PLANET MUG,-12,1.25
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,1.25
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,0.85
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2.95
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,4.25


Negative quantities include customer cancellations and returns, but also operational adjustments such as losses, shortages and damaged stock.

These records should not contribute positive merchandise demand to the forecasting target.

In [93]:
zero_price_positive = retail_2009_2010[
    (retail_2009_2010["Price"] == 0)
    & (retail_2009_2010["Quantity"] > 0)
]

print("Rows:", len(zero_price_positive))
print("Missing Customer ID:",
      zero_price_positive["Customer ID"].isna().sum())
print("Missing Description:",
      zero_price_positive["Description"].isna().sum())

display(
    zero_price_positive[
        ["Invoice", "StockCode", "Description", "Quantity", "Price"]
    ].head(20)
)

Rows: 1566
Missing Customer ID: 1535
Missing Description: 1101


,Invoice,StockCode,Description,Quantity,Price
3161,489659,21350,NaN,230,0.0
3731,489781,84292,NaN,17,0.0
4674,489825,22076,6 RIBBONS EMPIRE,12,0.0
5904,489861,DOT,DOTCOM POSTAGE,1,0.0
6378,489882,35751C,NaN,12,0.0
6555,489898,79323G,NaN,954,0.0
6581,489903,21166,NaN,48,0.0
6781,489998,48185,DOOR MAT FAIRY CAKE,2,0.0
7204,490015,21982,NaN,467,0.0
9249,490123,84508B,NaN,184,0.0


Zero-price transactions are heavily associated with missing customer information, missing descriptions and internal stock movements.

Because the forecasting target is genuine merchandise-order demand, zero-price transactions will be excluded.

In [94]:
print(
    "2009-2010 exact duplicates:",
    retail_2009_2010.duplicated().sum()
)

print(
    "2010-2011 exact duplicates:",
    retail_2010_2011.duplicated().sum()
)

2009-2010 exact duplicates: 6865
2010-2011 exact duplicates: 5268


In [95]:
retail_2009_2010[
    retail_2009_2010.duplicated(keep=False)
].head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
367,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,2009-12-01 11:34:00,0.65,16329.0,United Kingdom
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
383,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom
384,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,2009-12-01 11:34:00,0.65,16329.0,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom


Exact duplicates would inflate unit demand if retained.

Only fully duplicated transaction rows will be removed. Legitimate repeated purchases remain because at least one transaction field differs.

In [96]:
retail_2009_2010 = retail_2009_2010.drop_duplicates().copy()
retail_2010_2011 = retail_2010_2011.drop_duplicates().copy()

In [97]:
print(
    retail_2009_2010["InvoiceDate"].min(),
    "to",
    retail_2009_2010["InvoiceDate"].max()
)

print(
    retail_2010_2011["InvoiceDate"].min(),
    "to",
    retail_2010_2011["InvoiceDate"].max()
)

2009-12-01 07:45:00 to 2010-12-09 20:01:00
2010-12-01 08:26:00 to 2011-12-09 12:50:00


The two source worksheets overlap between 1 December and 9 December 2010.

The overlapping records were checked and found to be duplicated across both worksheets, so they must only be retained once.

In [99]:
retail_2010_2011 = retail_2010_2011[
    retail_2010_2011["InvoiceDate"] > "2010-12-09 23:59:59"
].copy()

retail_raw_combined = pd.concat(
    [retail_2009_2010, retail_2010_2011],
    ignore_index=True
)

print("Combined rows:", len(retail_raw_combined))
print("Duplicates:", retail_raw_combined.duplicated().sum())

Combined rows: 1033036
Duplicates: 0


In [103]:
retail_raw_combined = retail_raw_combined.reset_index(drop=True)
retail_raw_combined["row_id"] = retail_raw_combined.index

retail_raw_combined["invoice_day"] = (
    retail_raw_combined["InvoiceDate"].dt.date
)

positive_rows = retail_raw_combined[
    (retail_raw_combined["Quantity"] > 0)
    & (retail_raw_combined["Price"] > 0)
].copy()

cancel_rows = retail_raw_combined[
    (retail_raw_combined["Quantity"] < 0)
    & (retail_raw_combined["Price"] > 0)
    & (
        retail_raw_combined["Invoice"]
        .astype(str)
        .str.startswith("C")
    )
].copy()

positive_rows["match_quantity"] = positive_rows["Quantity"]
cancel_rows["match_quantity"] = cancel_rows["Quantity"].abs()

confirmed_reversals = cancel_rows.merge(
    positive_rows,
    on=[
        "StockCode",
        "Customer ID",
        "Price",
        "match_quantity",
        "invoice_day"
    ],
    suffixes=("_cancel", "_sale")
)

confirmed_reversals = confirmed_reversals[
    confirmed_reversals["InvoiceDate_sale"]
    <= confirmed_reversals["InvoiceDate_cancel"]
].copy()

print(
    "Confirmed reversal matches:",
    len(confirmed_reversals)
)

print(
    "Unique positive sales reversed:",
    confirmed_reversals["row_id_sale"].nunique()
)

Confirmed reversal matches: 1482
Unique positive sales reversed: 1449


Some very large positive transactions were followed minutes later by matching cancellation invoices.

For example, quantities of more than 70,000 units were fully reversed on the same day.

If these positive rows were retained while their negative counterparts were removed, the forecasting dataset would contain artificial demand spikes.

The notebook therefore removes the original positive transaction only where a same-day cancellation matches the same customer, product, quantity and price.

In [105]:
NON_PRODUCT_CODES = [
    "DOT",
    "M",
    "D",
    "AMAZONFEE",
    "B"
]

print(NON_PRODUCT_CODES)

['DOT', 'M', 'D', 'AMAZONFEE', 'B']


Several stock codes represent operational or accounting activity rather than merchandise:

- `DOT` — Dotcom postage
- `M` — Manual adjustment
- `D` — Discount
- `AMAZONFEE` — Amazon fee
- `B` — Bad debt adjustment

These records are excluded because they are not forecastable product demand.

In [108]:
reversed_sale_ids = (
    confirmed_reversals["row_id_sale"]
    .drop_duplicates()
)

retail_final = retail_raw_combined[
    (retail_raw_combined["Quantity"] > 0)
    & (retail_raw_combined["Price"] > 0)
    & (~retail_raw_combined["row_id"].isin(reversed_sale_ids))
    & (
        ~retail_raw_combined["StockCode"]
        .astype(str)
        .isin(NON_PRODUCT_CODES)
    )
].copy()

retail_final = retail_final.drop_duplicates(
    subset=[
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "Price",
        "Customer ID",
        "Country"
    ]
)

retail_final = retail_final.rename(columns={
    "Invoice": "invoice",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "Price": "unit_price",
    "Customer ID": "customer_id",
    "Country": "country"
})

retail_final["gross_order_value"] = (
    retail_final["quantity"]
    * retail_final["unit_price"]
)

retail_final = retail_final.drop(
    columns=["row_id", "invoice_day"],
    errors="ignore"
)

In [109]:
print("Final rows:", len(retail_final))
print("Exact duplicates:", retail_final.duplicated().sum())
print("Non-positive quantity:", (retail_final["quantity"] <= 0).sum())
print("Non-positive price:", (retail_final["unit_price"] <= 0).sum())

print("Products:", retail_final["stock_code"].nunique())
print("Customers:", retail_final["customer_id"].nunique())
print("Countries:", retail_final["country"].nunique())

print(
    "Date range:",
    retail_final["invoice_date"].min(),
    "to",
    retail_final["invoice_date"].max()
)

Final rows: 1004262
Exact duplicates: 0
Non-positive quantity: 0
Non-positive price: 0
Products: 4907
Customers: 5861
Countries: 43
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00


In [110]:
assert retail_final.duplicated().sum() == 0
assert (retail_final["quantity"] <= 0).sum() == 0
assert (retail_final["unit_price"] <= 0).sum() == 0

In [111]:
retail_final.nlargest(
    10,
    "quantity"
)[
    [
        "invoice",
        "stock_code",
        "description",
        "quantity",
        "unit_price",
        "gross_order_value",
        "country",
        "invoice_date"
    ]
]

,invoice,stock_code,description,quantity,unit_price,gross_order_value,country,invoice_date
89849,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,0.10,1915.2,Denmark,2010-02-15 11:57:00
125789,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,0.10,1296.0,Denmark,2010-03-17 13:09:00
125791,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,0.10,1296.0,Denmark,2010-03-17 13:09:00
125792,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,0.10,1274.4,Denmark,2010-03-17 13:09:00
125790,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,0.10,1248.0,Denmark,2010-03-17 13:09:00
133513,502269,21984,PACK OF 12 PINK PAISLEY TISSUES,10000,0.25,2500.0,United Kingdom,2010-03-23 15:36:00
133514,502269,21982,PACK OF 12 SUKI TISSUES,10000,0.25,2500.0,United Kingdom,2010-03-23 15:36:00
133515,502269,21980,PACK OF 12 RED SPOTTY TISSUES,10000,0.25,2500.0,United Kingdom,2010-03-23 15:36:00
133516,502269,21981,PACK OF 12 WOODLAND TISSUES,10000,0.25,2500.0,United Kingdom,2010-03-23 15:36:00
92646,498152,85220,SMALL FAIRY CAKE FRIDGE MAGNETS,9456,0.30,2836.8,Denmark,2010-02-17 10:51:00


High-volume transactions were not removed solely because they were statistical outliers.

After removing confirmed same-day reversals, the largest remaining transactions appeared consistent with legitimate bulk customer orders.

This avoids artificially smoothing real demand.

In [112]:
OUTPUT_PATH = DATA_CLEAN / "retail_transactions_clean.csv"

retail_final.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Saved to: {OUTPUT_PATH}")

Saved to: ..\data\cleaned\retail_transactions_clean.csv


## Cleaning Summary

The raw Online Retail II data was transformed into a transaction-level dataset suitable for demand analysis and forecasting.

Key decisions:

- removed exact duplicate transaction rows
- resolved the duplicated December 2010 overlap between source worksheets
- excluded negative quantities representing cancellations, returns and operational adjustments
- excluded zero and negative prices
- removed positive transactions that were fully reversed by matching same-day cancellations
- excluded non-merchandise transaction codes such as postage, fees, discounts and manual adjustments
- retained transactions with missing Customer IDs where valid product, quantity and date information remained
- retained legitimate high-volume orders rather than removing statistical outliers automatically

The resulting dataset represents **positive merchandise-order demand** and will be used in the exploratory analysis and forecasting stages.